# SemanticConsistencyLoss Multi-View Consistency Debug

Notebook per il debug visuale step-by-step della Multi-View Consistency Loss nella classe `SemanticConsistencyLoss` definita in `mapanything/train/losses.py`.

Obiettivi del notebook:
- verificare la proiezione geometrica dei punti 3D
- controllare l'occlusion check
- ispezionare le feature semantiche tramite PCA
- contare i pixel con corrispondenze valide
- individuare dove fallisce il matching

Il notebook si aspetta un checkpoint salvato con almeno:
- `batch`: lista di dizionari, uno per view
- `preds`: lista di dizionari, uno per view

Chiavi minime richieste per il debug selezionato:
- `batch[i]`: `pts3d`, `pts3d_cam`, `semantics`, `camera_pose`, `camera_intrinsics`
- `preds[i]`: `semantics`, `sem_conf`, `pts3d`

Se il key `image` non e' presente, il notebook usa un fallback visivo basato sulla PCA delle semantic features.

In [9]:
# ==== CONFIGURAZIONE INIZIALE ====
BATCH_IDX = 0
VIEW_I = 0
VIEW_J = 1
GRID_STEP = 8
CHECKPOINT_PATH = '/scratch2/nico/distillation/output/distillation_loss_full_dataset/checkpoints/checkpoint_encoder_10000.pth'
DEVICE = 'cuda'
PCA_COMPONENTS = 3
OCCLUSION_THRESHOLD = 0.1
MIN_CONF = 1.0
IMAGE_FOLDER = '/scratch2/nico/distillation/dataset/blendedmbs_picked/overfit/action_figure'
number_of_views = 4
input_size = 224

# Figure helpers
FIGSIZE_BASE = (18, 5)
DPI = 120

print('Config loaded')

Config loaded


In [8]:
import math
from pathlib import Path

from typing import Tuple
import os


import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.decomposition import PCA

from IPython.display import display, Markdown

from mapanything.models import MapAnything
from mapanything.train.losses import SemanticConsistencyLoss
from mapanything.utils.geometry import closed_form_pose_inverse, geotrf
from mapanything.utils.image import load_images


# %matplotlib inline

plt.style.use('seaborn-v0_8-whitegrid')
torch.set_grad_enabled(False)

def ensure_device(value, device):
    if torch.is_tensor(value):
        return value.to(device)
    return value

def select_sample(view_dict, batch_idx, device):
    sampled = {}
    for key, value in view_dict.items():
        if torch.is_tensor(value) and value.dim() > 0 and value.shape[0] > batch_idx:
            sampled[key] = value[batch_idx:batch_idx + 1].to(device)
        else:
            sampled[key] = ensure_device(value, device)
    return sampled

def tensor_stats(tensor, name):
    if not torch.is_tensor(tensor):
        return f'{name}: non-tensor value'
    with torch.no_grad():
        data = tensor.detach().float()
        finite = torch.isfinite(data)
        if finite.any():
            vals = data[finite]
            return (
                f'{name}: shape={tuple(tensor.shape)} '
                f'min={vals.min().item():.4f} max={vals.max().item():.4f} '
                f'mean={vals.mean().item():.4f} std={vals.std().item():.4f}'
            )
        return f'{name}: shape={tuple(tensor.shape)} no finite values'

def to_numpy_image(tensor):
    if not torch.is_tensor(tensor):
        return tensor
    data = tensor.detach().cpu()
    if data.dim() == 4:
        data = data[0]
    if data.dim() == 3 and data.shape[0] in (1, 3):
        data = data.permute(1, 2, 0)
    data = data.float().numpy()
    if data.ndim == 2:
        data = np.repeat(data[..., None], 3, axis=2)
    data = np.nan_to_num(data)
    data_min = data.min()
    data_max = data.max()
    if data_max - data_min > 1e-8:
        data = (data - data_min) / (data_max - data_min)
    data = (255.0 * np.clip(data, 0.0, 1.0)).astype(np.uint8)
    return data

def find_image_tensor(view_dict):
    for key in ('image', 'rgb', 'imgs', 'images', 'color'):
        if key in view_dict and torch.is_tensor(view_dict[key]):
            return view_dict[key]
    return None

def apply_pca_to_features(feat, n_components=3):
    """
    Input: (B, C, H, W)
    Output: (B, 3, H, W) in [0, 255]
    """
    if feat.dim() != 4:
        raise ValueError(f'Expected feat to have shape (B, C, H, W), got {tuple(feat.shape)}')
    b, c, h, w = feat.shape
    feat_cpu = feat.detach().float().cpu().permute(0, 2, 3, 1).reshape(-1, c).numpy()
    feat_cpu = np.nan_to_num(feat_cpu)
    n_fit_components = min(n_components, c, feat_cpu.shape[0])
    if n_fit_components < 1:
        raise ValueError('Cannot run PCA on an empty tensor')
    pca = PCA(n_components=n_fit_components)
    reduced = pca.fit_transform(feat_cpu)
    if reduced.shape[1] < 3:
        pad = np.zeros((reduced.shape[0], 3 - reduced.shape[1]), dtype=reduced.dtype)
        reduced = np.concatenate([reduced, pad], axis=1)
    reduced = reduced.reshape(b, h, w, 3)
    normalized = np.empty_like(reduced)
    for bi in range(b):
        img = reduced[bi]
        img_min = img.min(axis=(0, 1), keepdims=True)
        img_max = img.max(axis=(0, 1), keepdims=True)
        denom = np.maximum(img_max - img_min, 1e-8)
        normalized[bi] = (img - img_min) / denom
    normalized = (normalized * 255.0).clip(0, 255).astype(np.uint8)
    return torch.from_numpy(normalized).permute(0, 3, 1, 2)

def draw_points_on_image(img, coords_2d, colors=None, radius=5):
    """
    Draw 2D points on top of an image.
    coords_2d: (N, 2) in pixel coordinates as (x, y).
    colors: (N, 3) RGB or None for default red.
    """
    canvas = to_numpy_image(img).copy()
    if canvas.shape[2] == 1:
        canvas = np.repeat(canvas, 3, axis=2)
    canvas_bgr = cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR)
    coords_2d = np.asarray(coords_2d)
    if colors is None:
        colors = np.tile(np.array([[255, 0, 0]], dtype=np.uint8), (len(coords_2d), 1))
    else:
        colors = np.asarray(colors).astype(np.uint8)
    for (x, y), color in zip(coords_2d, colors):
        xi = int(round(float(x)))
        yi = int(round(float(y)))
        if 0 <= yi < canvas_bgr.shape[0] and 0 <= xi < canvas_bgr.shape[1]:
            cv2.circle(canvas_bgr, (xi, yi), radius, tuple(int(c) for c in color[::-1]), -1)
    return cv2.cvtColor(canvas_bgr, cv2.COLOR_BGR2RGB)

def compute_cosine_similarity(feat1, feat2):
    if feat1.shape != feat2.shape:
        raise ValueError(f'Shape mismatch: {tuple(feat1.shape)} vs {tuple(feat2.shape)}')
    feat1_n = F.normalize(feat1, dim=1)
    feat2_n = F.normalize(feat2, dim=1)
    return torch.sum(feat1_n * feat2_n, dim=1)

def select_grid_coords(height, width, step):
    ys = np.arange(0, height, step, dtype=np.int64)
    xs = np.arange(0, width, step, dtype=np.int64)
    yy, xx = np.meshgrid(ys, xs, indexing='ij')
    coords = np.stack([xx.reshape(-1), yy.reshape(-1)], axis=1)
    return coords

def summarize_mask(mask):
    mask_f = mask.float()
    return {
        'count': int(mask.numel()),
        'true': int(mask.sum().item()),
        'ratio': float(mask_f.mean().item()) if mask.numel() else 0.0,
    }

def load_encoder_checkpoint(
    model_without_ddp,
    checkpoint_path: str,
    device: torch.device,
) -> Tuple[int, float]:
    """
    Load checkpoint containing only ENCODER trainable components.
    
    Loads:
        - dpt_feature_head_2 (student encoder)
        - sam2_compat (student encoder compatibility layer)
        - unfrozen info_sharing blocks (multi-view transformer)
        - unfrozen DINOv2 encoder blocks
    
    Args:
        model_without_ddp: Model without DDP wrapper
        checkpoint_path: Path to encoder checkpoint
        device: Device to load checkpoint on
        args: Training arguments (optional, not required for loading)
    
    Returns:
        (start_epoch, best_val_loss) tuple from checkpoint
    """
    print(f"[LOAD] Loading encoder checkpoint: {checkpoint_path}")
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    print(f"[DEBUG] Checkpoint content: {ckpt.keys()}")
    
    # Load encoder head
    if "dpt_feature_head_2" in ckpt:
        model_without_ddp.dpt_feature_head_2.load_state_dict(ckpt["dpt_feature_head_2"])
        print("[INFO] Loaded dpt_feature_head_2")
    else:
        print("[WARN] dpt_feature_head_2 not found in encoder checkpoint!")
    
    # Load sam2_compat if present
    if "sam2_compat" in ckpt and hasattr(model_without_ddp, "sam2_compat"):
        model_without_ddp.sam2_compat.load_state_dict(ckpt["sam2_compat"])
        print("[INFO] Loaded sam2_compat")
    elif hasattr(model_without_ddp, "sam2_compat"):
        print("[INFO] sam2_compat not in encoder checkpoint; using random initialization")
    
    # Load info_sharing blocks (NO dipendenza da args)
    if "info_sharing_blocks" in ckpt and hasattr(model_without_ddp, "info_sharing"):
        info = model_without_ddp.info_sharing
        blocks = getattr(info, "self_attention_blocks", None)
        if blocks:
            for idx, state_dict in ckpt["info_sharing_blocks"].items():
                if idx < len(blocks):
                    try:
                        blocks[idx].load_state_dict(state_dict)
                    except Exception as e:
                        print(f"[WARN] Failed loading info_sharing block {idx}: {e}")
            print(f"[INFO] Restored {len(ckpt['info_sharing_blocks'])} info_sharing blocks")
        
        if "info_sharing_wrappers" in ckpt:
            for name, data in ckpt["info_sharing_wrappers"].items():
                try:
                    param = dict(info.named_parameters())[name]
                    param.data.copy_(data)
                except Exception as e:
                    print(f"[WARN] Failed loading wrapper param {name}: {e}")
            print(f"[INFO] Restored {len(ckpt['info_sharing_wrappers'])} info_sharing wrapper params")
    
    # Load DINOv2 blocks (NO dipendenza da args)
    if "dino_encoder_blocks" in ckpt and hasattr(model_without_ddp, "encoder"):
        dino_encoder = model_without_ddp.encoder
        dino_model = dino_encoder.model if hasattr(dino_encoder, "model") else dino_encoder
        blocks = getattr(dino_model, "blocks", None)
        if blocks:
            for idx, state_dict in ckpt["dino_encoder_blocks"].items():
                if idx < len(blocks):
                    try:
                        blocks[idx].load_state_dict(state_dict)
                    except Exception as e:
                        print(f"[WARN] Failed loading DINOv2 block {idx}: {e}")
            print(f"[INFO] Restored {len(ckpt['dino_encoder_blocks'])} DINOv2 blocks")
        
        if "dino_encoder_wrappers" in ckpt:
            for name, data in ckpt["dino_encoder_wrappers"].items():
                try:
                    param = dict(dino_model.named_parameters())[name]
                    param.data.copy_(data)
                except Exception as e:
                    print(f"[WARN] Failed loading wrapper param {name}: {e}")
            print(f"[INFO] Restored {len(ckpt['dino_encoder_wrappers'])} DINOv2 wrapper params")
    
    start_epoch = ckpt.get("epoch", 0) + 1
    best_val_loss = ckpt.get("best_val_loss", float("inf"))
    
    return start_epoch, best_val_loss

print('Helpers ready')

Helpers ready


In [4]:
# ==== LOAD CHECKPOINT ====
checkpoint_path = Path(CHECKPOINT_PATH)
image_folder = Path(IMAGE_FOLDER)
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

model = MapAnything.from_pretrained(
        "facebook/map-anything",
        revision="562de9ff7077addd5780415661c5fb031eb8003e",
        strict=False,
    ).to(DEVICE)

_, _ = load_encoder_checkpoint(
        model_without_ddp=model,
        checkpoint_path=checkpoint_path,
        device=DEVICE,
    )

# Load images
print(f"Loading images from: {image_folder}")

supported_extensions = (".jpg", ".jpeg", ".png", ".heic", ".heif")
image_names = sorted(
    name
    for name in os.listdir(image_folder)
    if name.lower().endswith(supported_extensions)
)
views = [
    os.path.join(image_folder, name)
    for name in image_names[: number_of_views]
]

views = load_images(
        views,
        resize_mode="square",
        size=input_size,
        norm_type="dinov2",
    )
print(f"Loaded {len(views)} views")

outputs = model.infer(
        views, memory_efficient_inference=False
    )

batch = checkpoint['batch']
preds = checkpoint['preds']

if not isinstance(batch, list) or not isinstance(preds, list):
    raise TypeError('batch and preds must be lists of per-view dictionaries')
if len(batch) != len(preds):
    raise ValueError(f'batch and preds length mismatch: {len(batch)} vs {len(preds)}')

n_views = len(batch)
if n_views < 2:
    raise ValueError('Need at least two views for multi-view consistency debug')

view_i = int(np.clip(VIEW_I, 0, n_views - 1))
view_j = int(np.clip(VIEW_J, 0, n_views - 1))
if view_i == view_j and n_views > 1:
    view_j = (view_i + 1) % n_views

batch_i = select_sample(batch[view_i], BATCH_IDX, DEVICE)
batch_j = select_sample(batch[view_j], BATCH_IDX, DEVICE)
pred_i = select_sample(preds[view_i], BATCH_IDX, DEVICE)
pred_j = select_sample(preds[view_j], BATCH_IDX, DEVICE)

required_batch_keys = {'pts3d', 'pts3d_cam', 'semantics', 'camera_pose', 'camera_intrinsics'}
required_pred_keys = {'semantics', 'sem_conf', 'pts3d'}
missing_batch = sorted(required_batch_keys - set(batch_i.keys()))
missing_pred = sorted(required_pred_keys - set(pred_i.keys()))
if missing_batch:
    raise KeyError(f'Missing keys in batch[{view_i}]: {missing_batch}')
if missing_pred:
    raise KeyError(f'Missing keys in preds[{view_i}]: {missing_pred}')

loss_obj = SemanticConsistencyLoss(
    distillation_weight=1.0,
    consistency_weight=1.0,
    min_conf=MIN_CONF,
    occlusion_thresh=OCCLUSION_THRESHOLD,
)

display(Markdown(f'Loaded checkpoint: `{checkpoint_path}`'))
display(Markdown(f'Views available: {n_views}. Debug pair: view {view_i} -> view {view_j}. Batch idx: {BATCH_IDX}'))
print(tensor_stats(batch_i['pts3d'], 'batch_i["pts3d"]'))
print(tensor_stats(pred_i['semantics'], 'pred_i["semantics"]'))

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL omegaconf.dictconfig.DictConfig was not an allowed global by default. Please use `torch.serialization.add_safe_globals([DictConfig])` or the `torch.serialization.safe_globals([DictConfig])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default

In [ ]:
# ==== SEZIONE 2: PREPARAZIONE DATI ====
anchor_img = find_image_tensor(batch_i)
target_img = find_image_tensor(batch_j)

if anchor_img is None:
    anchor_img = batch_i['semantics']
if target_img is None:
    target_img = batch_j['semantics']

anchor_pts_world = batch_i['pts3d']
anchor_pts_cam = batch_i['pts3d_cam']
anchor_feat = F.interpolate(pred_i['semantics'], size=anchor_pts_world.shape[1:3], mode='bilinear', align_corners=False)
anchor_conf = pred_i['sem_conf']
if anchor_conf.shape[2:] != anchor_pts_world.shape[1:3]:
    anchor_conf = F.interpolate(anchor_conf, size=anchor_pts_world.shape[1:3], mode='bilinear', align_corners=False)
anchor_conf = anchor_conf.clamp_min(MIN_CONF)

tgt_feat_upsampled = F.interpolate(pred_j['semantics'], size=anchor_pts_world.shape[1:3], mode='bilinear', align_corners=False)
tgt_conf = pred_j['sem_conf']
if tgt_conf.shape[2:] != anchor_pts_world.shape[1:3]:
    tgt_conf = F.interpolate(tgt_conf, size=anchor_pts_world.shape[1:3], mode='bilinear', align_corners=False)
tgt_conf = tgt_conf.clamp_min(MIN_CONF)

display(Markdown('### Shape and statistics'))
for name, tensor in [
    ('anchor_pts_world', anchor_pts_world),
    ('anchor_feat', anchor_feat),
    ('anchor_conf', anchor_conf),
    ('tgt_feat_upsampled', tgt_feat_upsampled),
    ('tgt_conf', tgt_conf),
]:
    print(tensor_stats(tensor, name))

anchor_img_np = to_numpy_image(anchor_img)
target_img_np = to_numpy_image(target_img)
anchor_pca = apply_pca_to_features(anchor_feat, n_components=PCA_COMPONENTS)
tgt_pca = apply_pca_to_features(tgt_feat_upsampled, n_components=PCA_COMPONENTS)

print(f'Anchor image shape: {anchor_img_np.shape}')
print(f'Target image shape: {target_img_np.shape}')
print(f'PCA anchor feature shape: {tuple(anchor_pca.shape)}')
print(f'PCA target feature shape: {tuple(tgt_pca.shape)}')

In [ ]:
# ==== SEZIONE 3: VISUALIZZAZIONE PER-PIXEL PCA ====
h_orig, w_orig = anchor_pts_world.shape[1:3]
grid_coords = select_grid_coords(h_orig, w_orig, GRID_STEP)
grid_x = grid_coords[:, 0]
grid_y = grid_coords[:, 1]
selected_anchor_colors = anchor_pca[0, :, grid_y, grid_x].permute(1, 0).cpu().numpy()
selected_anchor_colors = np.clip(selected_anchor_colors, 0, 255).astype(np.uint8)

overlay_anchor = draw_points_on_image(anchor_img_np, grid_coords, colors=selected_anchor_colors, radius=4)
anchor_pca_rgb = anchor_pca[0].permute(1, 2, 0).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE_BASE, dpi=DPI)
axes[0].imshow(overlay_anchor)
axes[0].set_title('A. Anchor image with sampled points')
axes[0].axis('off')

axes[1].imshow(anchor_pca_rgb)
axes[1].set_title('B. Anchor semantic PCA RGB')
axes[1].axis('off')

axes[2].imshow(np.zeros((64, 64, 3), dtype=np.uint8))
axes[2].scatter(
    np.arange(len(selected_anchor_colors)),
    np.zeros(len(selected_anchor_colors)),
    c=selected_anchor_colors / 255.0,
    s=40,
)
axes[2].set_xlim(-1, len(selected_anchor_colors))
axes[2].set_ylim(-1, 1)
axes[2].set_title('C. PCA color per sampled point')
axes[2].axis('off')
plt.tight_layout()

display(Markdown(f'Sampled points: {len(grid_coords)} using grid_step={GRID_STEP}'))

In [ ]:
# ==== SEZIONE 4: PROIEZIONE (STEP 6) ====
grid, z_proj, fov_mask = loss_obj.project_to_view(
    anchor_pts_world,
    batch_j['camera_pose'],
    batch_j['camera_intrinsics'],
    anchor_pts_world.shape[1:3],
)

grid_np = grid[0].detach().cpu().numpy()
z_proj_np = z_proj[0, 0].detach().cpu().numpy()
fov_mask_np = fov_mask[0, 0].detach().cpu().numpy().astype(np.float32)
valid_fov_count = int(fov_mask.sum().item())
total_count = int(fov_mask.numel())

projected_coords = np.stack([
    (grid_np[..., 0] + 1.0) * 0.5 * (w_orig - 1),
    (grid_np[..., 1] + 1.0) * 0.5 * (h_orig - 1),
], axis=-1).reshape(-1, 2)
valid_projected = fov_mask_np.reshape(-1) > 0.5

target_overlay = draw_points_on_image(target_img_np, projected_coords[::max(1, GRID_STEP)], colors=None, radius=3)

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE_BASE, dpi=DPI)
axes[0].imshow(target_overlay)
axes[0].set_title('A. Target image with projections')
axes[0].axis('off')

im1 = axes[1].imshow(fov_mask_np, cmap='coolwarm', vmin=0, vmax=1)
axes[1].set_title('B. FOV mask')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(z_proj_np, cmap='viridis')
axes[2].set_title('C. Projected depth z_proj')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()

display(Markdown(f'FOV points: {valid_fov_count} / {total_count} ({100.0 * valid_fov_count / max(total_count, 1):.2f}%)'))

In [ ]:
# ==== SEZIONE 5: CAMPIONAMENTO (STEP 7) ====
sampled_feat = F.grid_sample(tgt_feat_upsampled, grid, align_corners=True, padding_mode='border')
sampled_conf = F.grid_sample(tgt_conf, grid, align_corners=True, padding_mode='border').clamp_min(MIN_CONF)
sampled_feat_pca = apply_pca_to_features(sampled_feat, n_components=PCA_COMPONENTS)
sampled_feat_pca_rgb = sampled_feat_pca[0].permute(1, 2, 0).cpu().numpy()
sampled_conf_np = sampled_conf[0, 0].detach().cpu().numpy()

sampled_anchor_colors = anchor_pca[0, :, grid_y, grid_x].permute(1, 0).cpu().numpy()
sampled_tgt_colors = sampled_feat_pca[0, :, grid_y, grid_x].permute(1, 0).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE_BASE, dpi=DPI)
axes[0].imshow(sampled_feat_pca_rgb)
axes[0].set_title('A. PCA of sampled target features')
axes[0].axis('off')

im1 = axes[1].imshow(sampled_conf_np, cmap='magma')
axes[1].set_title('B. Sampled confidence')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

for idx, (ac, tc) in enumerate(zip(sampled_anchor_colors[:100], sampled_tgt_colors[:100])):
    x = idx
    axes[2].scatter(x, 1, c=[ac / 255.0], s=30, marker='s')
    axes[2].scatter(x, 0, c=[tc / 255.0], s=30, marker='s')
axes[2].set_title('C. Anchor color (top) vs sampled target color (bottom)')
axes[2].set_xlim(-1, min(100, len(sampled_anchor_colors)))
axes[2].set_ylim(-1, 2)
axes[2].axis('off')
plt.tight_layout()

display(Markdown('The sampled target features should look similar to the anchor features at valid correspondences.'))

In [ ]:
# ==== SEZIONE 6: OCCLUSION CHECK (STEP 8) ====
gt_pts_cam_j_high = batch_j['pts3d_cam']
tgt_depth_map = gt_pts_cam_j_high[..., 2:3]
sampled_tgt_depth = F.grid_sample(tgt_depth_map.permute(0, 3, 1, 2), grid, align_corners=True, padding_mode='border')
depth_diff = torch.abs(z_proj - sampled_tgt_depth)
occ_mask = depth_diff < OCCLUSION_THRESHOLD
valid_mask = fov_mask & occ_mask

fov_total = fov_mask.sum().item()
occ_total = occ_mask.sum().item()
valid_total = valid_mask.sum().item()
fov_pass_ratio = float(valid_total / max(fov_total, 1))

depth_diff_np = depth_diff[0, 0].detach().cpu().numpy()
occ_mask_np = occ_mask[0, 0].detach().cpu().numpy().astype(np.float32)
valid_mask_np = valid_mask[0, 0].detach().cpu().numpy().astype(np.float32)

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE_BASE, dpi=DPI)
im0 = axes[0].imshow(depth_diff_np, cmap='inferno')
axes[0].set_title('|z_proj - sampled_tgt_depth|')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(occ_mask_np, cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_title('Occlusion mask')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(valid_mask_np, cmap='RdYlGn', vmin=0, vmax=1)
axes[2].set_title('Final valid mask')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()

display(Markdown(
    f'FOV pass ratio after occlusion check: {100.0 * fov_pass_ratio:.2f}% (valid / FOV)'
))
print(tensor_stats(depth_diff, 'depth_diff'))

In [ ]:
# ==== SEZIONE 7: ACCUMULO (STEP 9) ====
sum_weighted_feat = anchor_feat * anchor_conf
sum_conf = anchor_conf.clone()
has_matches_mask = torch.zeros((1, h_orig, w_orig), device=anchor_feat.device, dtype=torch.bool)
has_matches_mask = has_matches_mask | valid_mask.squeeze(1)
s_feat = sampled_feat * valid_mask.float()
s_conf = sampled_conf * valid_mask.float()
sum_weighted_feat = sum_weighted_feat + (s_feat * s_conf)
sum_conf = sum_conf + s_conf

has_matches_np = has_matches_mask[0].detach().cpu().numpy().astype(np.float32)
sum_conf_np = sum_conf[0, 0].detach().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE_BASE, dpi=DPI)
axes[0].imshow(valid_mask_np, cmap='RdYlGn', vmin=0, vmax=1)
axes[0].set_title('A. Pair valid mask')
axes[0].axis('off')

axes[1].imshow(has_matches_np, cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_title('B. has_matches_mask')
axes[1].axis('off')

im2 = axes[2].imshow(sum_conf_np, cmap='plasma')
axes[2].set_title('C. Accumulated confidence')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()

display(Markdown(
    f'Pixels with at least one valid match in this pair: {100.0 * float(has_matches_mask.float().mean().item()):.2f}%'
))

In [ ]:
# ==== SEZIONE 8: MEDIA PONDERATA (STEP 10) ====
mean_feat = sum_weighted_feat / (sum_conf + 1e-6)
mean_feat = F.normalize(mean_feat, p=2, dim=1)
target_feat = mean_feat.detach()
mean_feat_pca = apply_pca_to_features(mean_feat, n_components=PCA_COMPONENTS)
mean_feat_pca_rgb = mean_feat_pca[0].permute(1, 2, 0).cpu().numpy()

anchor_feat_norm = F.normalize(anchor_feat, p=2, dim=1)
mean_feat_norm = F.normalize(mean_feat, p=2, dim=1)

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE_BASE, dpi=DPI)
axes[0].imshow(mean_feat_pca_rgb)
axes[0].set_title('A. Mean feature PCA')
axes[0].axis('off')

sample_idx = np.arange(min(100, len(grid_coords)))
anchor_colors = anchor_pca[0, :, grid_y[sample_idx], grid_x[sample_idx]].permute(1, 0).cpu().numpy()
mean_colors = mean_feat_pca[0, :, grid_y[sample_idx], grid_x[sample_idx]].permute(1, 0).cpu().numpy()
for idx, (ac, mc) in enumerate(zip(anchor_colors, mean_colors)):
    axes[1].scatter(idx, 1, c=[ac / 255.0], s=30, marker='s')
    axes[1].scatter(idx, 0, c=[mc / 255.0], s=30, marker='s')
axes[1].set_title('B. Anchor (top) vs mean (bottom) colors')
axes[1].set_xlim(-1, len(sample_idx))
axes[1].set_ylim(-1, 2)
axes[1].axis('off')

sim_i = torch.sum(anchor_feat_norm * mean_feat_norm, dim=1)
sim_i_np = sim_i[0].detach().cpu().numpy()
im2 = axes[2].imshow(sim_i_np, cmap='coolwarm', vmin=-1, vmax=1)
axes[2].set_title('C. Cosine similarity map')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()

In [ ]:
# ==== SEZIONE 9: LOSS FINALE (STEP 11) ====
loss_i = 1.0 - sim_i
loss_i_np = loss_i[0].detach().cpu().numpy()
sim_i_np = sim_i[0].detach().cpu().numpy()

loss_mean = float(loss_i.mean().item())
loss_min = float(loss_i.min().item())
loss_max = float(loss_i.max().item())
loss_thr = 0.25
loss_high_count = int((loss_i > loss_thr).sum().item())
loss_total_count = int(loss_i.numel())

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE_BASE, dpi=DPI)
im0 = axes[0].imshow(sim_i_np, cmap='coolwarm', vmin=-1, vmax=1)
axes[0].set_title('A. Cosine similarity')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(loss_i_np, cmap='magma')
axes[1].set_title('B. Per-pixel loss')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

axes[2].hist(loss_i_np.reshape(-1), bins=50, color='steelblue', alpha=0.9)
axes[2].set_title('C. Loss histogram')
axes[2].set_xlabel('loss')
axes[2].set_ylabel('count')
plt.tight_layout()

display(Markdown(
    f'Average loss: {loss_mean:.6f} | min: {loss_min:.6f} | max: {loss_max:.6f} | ' +
    f'pixels with loss > {loss_thr}: {loss_high_count}/{loss_total_count}'
))

In [ ]:
# ==== SEZIONE 10: SUMMARY LOOP SU TUTTE LE COPPIE ====
def compute_pair_statistics(batch, preds, view_i, view_j, batch_idx, grid_step, device, loss_obj):
    bi = select_sample(batch[view_i], batch_idx, device)
    bj = select_sample(batch[view_j], batch_idx, device)
    pi = select_sample(preds[view_i], batch_idx, device)
    pj = select_sample(preds[view_j], batch_idx, device)

    anchor_pts_world_local = bi['pts3d']
    anchor_feat_local = F.interpolate(pi['semantics'], size=anchor_pts_world_local.shape[1:3], mode='bilinear', align_corners=False)
    anchor_conf_local = pi['sem_conf']
    if anchor_conf_local.shape[2:] != anchor_pts_world_local.shape[1:3]:
        anchor_conf_local = F.interpolate(anchor_conf_local, size=anchor_pts_world_local.shape[1:3], mode='bilinear', align_corners=False)
    anchor_conf_local = anchor_conf_local.clamp_min(MIN_CONF)

    tgt_feat_local = F.interpolate(pj['semantics'], size=anchor_pts_world_local.shape[1:3], mode='bilinear', align_corners=False)
    tgt_conf_local = pj['sem_conf']
    if tgt_conf_local.shape[2:] != anchor_pts_world_local.shape[1:3]:
        tgt_conf_local = F.interpolate(tgt_conf_local, size=anchor_pts_world_local.shape[1:3], mode='bilinear', align_corners=False)
    tgt_conf_local = tgt_conf_local.clamp_min(MIN_CONF)

    grid_local, z_proj_local, fov_mask_local = loss_obj.project_to_view(
        anchor_pts_world_local, bj['camera_pose'], bj['camera_intrinsics'], anchor_pts_world_local.shape[1:3]
    )
    sampled_feat_local = F.grid_sample(tgt_feat_local, grid_local, align_corners=True, padding_mode='border')
    sampled_conf_local = F.grid_sample(tgt_conf_local, grid_local, align_corners=True, padding_mode='border').clamp_min(MIN_CONF)
    sampled_depth_local = F.grid_sample(bj['pts3d_cam'][..., 2:3].permute(0, 3, 1, 2), grid_local, align_corners=True, padding_mode='border')
    occ_mask_local = torch.abs(z_proj_local - sampled_depth_local) < OCCLUSION_THRESHOLD
    valid_mask_local = fov_mask_local & occ_mask_local

    mean_feat_local = (anchor_feat_local * anchor_conf_local + sampled_feat_local * sampled_conf_local * valid_mask_local.float()) / (anchor_conf_local + sampled_conf_local * valid_mask_local.float() + 1e-6)
    mean_feat_local = F.normalize(mean_feat_local, p=2, dim=1)
    sim_local = compute_cosine_similarity(anchor_feat_local, mean_feat_local)
    loss_local = 1.0 - sim_local

    stats = {
        'view_i': view_i,
        'view_j': view_j,
        'fov_pct': float(fov_mask_local.float().mean().item()),
        'occ_pct': float(occ_mask_local.float().mean().item()),
        'valid_pct': float(valid_mask_local.float().mean().item()),
        'mean_loss': float(loss_local.mean().item()),
        'mean_sim': float(sim_local.mean().item()),
    }
    return stats, {
        'valid_mask': valid_mask_local[0, 0].detach().cpu().numpy(),
        'sim': sim_local[0].detach().cpu().numpy(),
        'loss': loss_local[0].detach().cpu().numpy(),
    }

all_stats = []
pair_images = {}
for ii in range(n_views):
    for jj in range(n_views):
        if ii == jj:
            continue
        stats, images = compute_pair_statistics(batch, preds, ii, jj, BATCH_IDX, GRID_STEP, DEVICE, loss_obj)
        all_stats.append(stats)
        pair_images[(ii, jj)] = images

header = '| view_i | view_j | valid % | fov % | occ % | mean sim | mean loss |\n'
header += '|---:|---:|---:|---:|---:|---:|---:|\n'
rows = []
for s in all_stats:
    rows.append(
        f'| {s["view_i"]} | {s["view_j"]} | {100.0 * s["valid_pct"]:.2f} | {100.0 * s["fov_pct"]:.2f} | {100.0 * s["occ_pct"]:.2f} | {s["mean_sim"]:.4f} | {s["mean_loss"]:.4f} |'
    )
display(Markdown('### Summary table'))
display(Markdown(header + '\n'.join(rows)))

fig, axes = plt.subplots(n_views, n_views, figsize=(3.2 * n_views, 3.2 * n_views), dpi=DPI)
if n_views == 1:
    axes = np.array([[axes]])
for ii in range(n_views):
    for jj in range(n_views):
        ax = axes[ii, jj]
        if ii == jj:
            ax.axis('off')
            ax.set_title(f'{ii} -> {jj}')
            continue
        img = pair_images[(ii, jj)]['valid_mask']
        ax.imshow(img, cmap='RdYlGn', vmin=0, vmax=1)
        st = next(s for s in all_stats if s['view_i'] == ii and s['view_j'] == jj)
        ax.set_title(f'{ii}->{jj} | valid {100.0 * st["valid_pct"]:.1f}% | loss {st["mean_loss"]:.3f}')
        ax.axis('off')
plt.tight_layout()

display(Markdown('The summary grid uses the same debug sampling step as the selected pair. Reduce `GRID_STEP` to inspect finer correspondences.'))